In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Beispiel-Vektoren
# König: Hat starke Komponenten in den ersten beiden Dimensionen (z.B. "männlich", "regal")
koenig = np.array([1, 1, 0, 0])
vergleichs_vektoren = {
    # Königin: Ähnlich wie König, aber mit einer Komponente in der dritten Dimension (z.B. "weiblich")
    "Königin": np.array([1, 1, 1, 0]),
    # Königin_2: Ähnliche Richtung wie König, aber mit größerer "Magnitude"
    "Königin_2": np.array([2, 2, 0.1, 0]),
    # Königin_3: Noch größere "Magnitude"
    "Königin_3": np.array([3, 3, 0.2, 0]),
    # Apfel: Zeigt in eine völlig andere Richtung (z.B. "Frucht", "Lebensmittel")
    "Apfel": np.array([0, 0, 10, 10])
}

In [2]:
def vektor_metriken(vektor_a: np.ndarray, vektor_b: np.ndarray) -> tuple[float, float, float, float]:
    
    if not isinstance(vektor_a, np.ndarray) or not isinstance(vektor_b, np.ndarray):
        raise ValueError("vektor_a und vektor_b müssen NumPy-Arrays sein.")
    if vektor_a.shape != vektor_b.shape:
        raise ValueError("vektor_a und vektor_b müssen die gleiche Form haben.")

    # Skalarprodukt (Dot Product): Misst die "Projektion" eines Vektors auf den anderen
    dot_product = np.dot(vektor_a, vektor_b)
    
    # Kosinus-Ähnlichkeit: Misst den Cosinus des Winkels zwischen den Vektoren
    # Formel: (A . B) / (||A|| * ||B||)
    norm_a = np.linalg.norm(vektor_a)
    norm_b = np.linalg.norm(vektor_b)
    kosinus = dot_product / (norm_a * norm_b) if (norm_a * norm_b) != 0 else 0
    
    # Euklidische Distanz: Geradliniger Abstand im Vektorraum
    euklidisch = np.linalg.norm(vektor_a - vektor_b)
    # Manhattan-Distanz: Summe der absoluten Differenzen der Koordinaten (City-Block-Distanz)
    manhattan = np.linalg.norm(vektor_a - vektor_b, ord=1)
    
    return dot_product, kosinus, euklidisch, manhattan

In [3]:
def vergleiche_vektoren(referenz_vektor: np.ndarray, vergleichs_vektoren_dict: dict[str, np.ndarray]) -> pd.DataFrame:

    if not isinstance(referenz_vektor, np.ndarray):
        raise ValueError("referenz_vektor muss ein NumPy-Array sein.")
    if not isinstance(vergleichs_vektoren_dict, dict):
        raise ValueError("vergleichs_vektoren_dict muss ein Dictionary sein.")

    ergebnisse = []
    for name, vektor in vergleichs_vektoren_dict.items():
        if not isinstance(vektor, np.ndarray):
            raise ValueError(f"Der Wert für '{name}' im vergleichs_vektoren_dict muss ein NumPy-Array sein.")
        dot_product, kosinus, euklidisch, manhattan = vektor_metriken(referenz_vektor, vektor)
        ergebnisse.append({
            "Vergleichs-Wort": name,
            "Skalarprodukt": dot_product,
            "Kosinus-Ähnlichkeit": kosinus,
            "Euklidische Distanz": euklidisch,
            "Manhattan-Distanz": manhattan
        })

    df = pd.DataFrame(ergebnisse)
    # Setze "Vergleichs-Wort" als Index für bessere Lesbarkeit
    df = df.set_index("Vergleichs-Wort")
    return df

In [4]:
ergebnisse_df = vergleiche_vektoren(koenig, vergleichs_vektoren)
print(ergebnisse_df)

                 Skalarprodukt  Kosinus-Ähnlichkeit  Euklidische Distanz  \
Vergleichs-Wort                                                            
Königin                    2.0             0.816497             1.000000   
Königin_2                  4.0             0.999376             1.417745   
Königin_3                  6.0             0.998891             2.835489   
Apfel                      0.0             0.000000            14.212670   

                 Manhattan-Distanz  
Vergleichs-Wort                     
Königin                        1.0  
Königin_2                      2.1  
Königin_3                      4.2  
Apfel                         22.0  
